In [1]:
# ==============================================================
# LLAMAINDEX + GEMINI + CHROMADB + TOOLS
# SINGLE-CELL JUPYTER NOTEBOOK PROGRAM
#
# Application: Intelligent Customer-Support Agent
# ==============================================================


# --------------------------------------------------------------
# 1. Install the required packages
# --------------------------------------------------------------

import sys
import subprocess
import os
from getpass import getpass

packages = [
    "llama-index",
    "llama-index-llms-gemini",
    "llama-index-embeddings-gemini",
    "llama-index-vector-stores-chroma",
    "chromadb",
    "google-generativeai"
]

print("Installing required packages...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        *packages
    ]
)

print("Packages installed successfully.\n")


Installing required packages...
Packages installed successfully.



In [2]:
# --------------------------------------------------------------
# 2. Import required libraries
# --------------------------------------------------------------

import chromadb

from datetime import datetime
from typing import Dict, Any

from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex
)

from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding

from llama_index.vector_stores.chroma import ChromaVectorStore

from llama_index.core.tools import (
    FunctionTool,
    QueryEngineTool
)

from llama_index.core.agent.workflow import FunctionAgent



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
#client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key

In [4]:
# --------------------------------------------------------------
# 4. Configure the Gemini LLM
# --------------------------------------------------------------

gemini_llm = Gemini(
    model="models/gemini-2.5-flash",
    api_key=api_key,
    temperature=0
)


# --------------------------------------------------------------
# 5. Configure the Gemini embedding model
# --------------------------------------------------------------

gemini_embedding_model = GeminiEmbedding(
    model_name="models/gemini-embedding-001",
    api_key=api_key
)


# Apply global LlamaIndex settings
Settings.llm = gemini_llm
Settings.embed_model = gemini_embedding_model

Settings.chunk_size = 256
Settings.chunk_overlap = 30


# --------------------------------------------------------------
# 6. Create company knowledge-base documents
# --------------------------------------------------------------

knowledge_documents = [

    Document(
        text="""
        Router Troubleshooting Policy

        If the router's red light is blinking, switch off the
        router and disconnect the power supply.

        Wait for thirty seconds, reconnect the power supply and
        restart the router.

        Verify that the fibre-optic cable is firmly connected.

        If the red light continues blinking after two restarts,
        contact technical support.
        """,
        metadata={
            "category": "router troubleshooting",
            "department": "technical support"
        }
    ),

    Document(
        text="""
        Slow Internet Troubleshooting

        If the internet connection is slow, first check the
        Wi-Fi signal strength.

        Move closer to the router and disconnect devices that
        are not currently being used.

        Pause large downloads and software updates.

        Restart both the modem and router and then perform a
        speed test.
        """,
        metadata={
            "category": "slow internet",
            "department": "technical support"
        }
    ),

    Document(
        text="""
        Product Warranty Policy

        Routers and modems include a one-year warranty from the
        original date of purchase.

        Customers must provide the original invoice and the
        device serial number for warranty replacement.

        Physical damage, water damage and unauthorized repairs
        are not covered by the warranty.
        """,
        metadata={
            "category": "warranty",
            "department": "customer service"
        }
    ),

    Document(
        text="""
        Password Reset Procedure

        Customers can reset their password by selecting
        Forgot Password on the login page.

        A password-reset link will be sent to the registered
        email address.

        The link expires after thirty minutes.

        A new password must contain at least eight characters,
        one uppercase letter, one lowercase letter and one number.
        """,
        metadata={
            "category": "password",
            "department": "account support"
        }
    ),

    Document(
        text="""
        Billing Support Policy

        For billing issues, customers must provide their customer
        ID, invoice number and transaction date.

        Billing complaints may include duplicate payments,
        incorrect charges, failed transactions and missing receipts.

        The billing team normally responds within two working days.
        """,
        metadata={
            "category": "billing",
            "department": "accounts"
        }
    ),

    Document(
        text="""
        Cancellation and Refund Policy

        A subscription can be cancelled through the customer
        account portal.

        A full refund is available when cancellation is requested
        within seven days of purchasing a new subscription,
        provided less than ten percent of the monthly data
        allowance has been used.

        Approved refunds are normally processed within five to
        seven working days.
        """,
        metadata={
            "category": "refund",
            "department": "accounts"
        }
    ),

    Document(
        text="""
        Customer-Support Working Hours

        Technical support is available from Monday to Saturday,
        between 9:00 a.m. and 6:00 p.m.

        Billing support is available from Monday to Friday,
        between 10:00 a.m. and 5:00 p.m.

        Network-outage complaints can be registered through the
        online customer portal at any time.
        """,
        metadata={
            "category": "support hours",
            "department": "customer service"
        }
    )
]

print(
    f"Created {len(knowledge_documents)} "
    "knowledge-base documents."
)


/tmp/ipykernel_8072/2092741237.py:5: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  gemini_llm = Gemini(
/tmp/ipykernel_8072/2092741237.py:16: DeprecationWarning: Call to deprecated class GeminiEmbedding. (Should use `llama-index-embeddings-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/embeddings/google_genai/Support for this package will be discontinued as of v0.4.2) -- Deprecated since version 0.4.2.
  gemini_embedding_model = GeminiEmbedding(


Created 7 knowledge-base documents.


In [13]:
# --------------------------------------------------------------
# 7. Create an in-memory ChromaDB database
# --------------------------------------------------------------

chroma_client = chromadb.EphemeralClient()

collection_name = "gemini_customer_support"

# Remove the old collection if the cell is rerun.
try:
    chroma_client.delete_collection(
        name=collection_name
    )
except Exception:
    pass


chroma_collection = chroma_client.create_collection(
    name=collection_name
)


chroma_vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)


storage_context = StorageContext.from_defaults(
    vector_store=chroma_vector_store
)





In [21]:
# --------------------------------------------------------------
# 8. Create Gemini embeddings and store them in ChromaDB
# --------------------------------------------------------------

print(
    "Creating Gemini embeddings and storing them in ChromaDB..."
)

index = VectorStoreIndex.from_documents(
    knowledge_documents,
    storage_context=storage_context,
    show_progress=True
)

print("ChromaDB vector index created successfully.\n")


# --------------------------------------------------------------
# 9. Create the RAG query engine
# --------------------------------------------------------------

#knowledge_query_engine = index.as_query_engine(
query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact",
    llm=gemini_llm
)

Creating Gemini embeddings and storing them in ChromaDB...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/7 [00:00<?, ?it/s]

ChromaDB vector index created successfully.



In [22]:
# --------------------------------------------------------------
# 10. Convert the RAG query engine into an agent tool
# --------------------------------------------------------------

knowledge_base_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,

    name="search_company_knowledge_base",

    description=(
        "Search the company knowledge base for information "
        "about routers, slow internet, passwords, warranties, "
        "refunds, billing and support hours. Use this tool for "
        "company policies and troubleshooting questions."
    )
)


# --------------------------------------------------------------
# 11. Create a simulated order database
# --------------------------------------------------------------

order_database = {

    "ORD1001": {
        "customer": "Arun",
        "product": "Wi-Fi 6 Router",
        "status": "Shipped",
        "expected_delivery": "20 July 2026"
    },

    "ORD1002": {
        "customer": "Meena",
        "product": "Fibre Modem",
        "status": "Processing",
        "expected_delivery": "22 July 2026"
    },

    "ORD1003": {
        "customer": "Karthik",
        "product": "Mesh Wi-Fi System",
        "status": "Delivered",
        "expected_delivery": "Delivered on 14 July 2026"
    }
}


# --------------------------------------------------------------
# 12. Tool 1: Check order status
# --------------------------------------------------------------

def check_order_status(order_id: str) -> str:
    """
    Check an order's current status.

    Args:
        order_id: Order ID such as ORD1001.

    Returns:
        Order status and delivery information.
    """

    normalized_order_id = order_id.strip().upper()

    order = order_database.get(normalized_order_id)

    if order is None:
        return (
            f"No order was found with ID "
            f"{normalized_order_id}. "
            "Please check the order number."
        )

    return (
        f"Order ID: {normalized_order_id}\n"
        f"Customer: {order['customer']}\n"
        f"Product: {order['product']}\n"
        f"Status: {order['status']}\n"
        f"Expected delivery: "
        f"{order['expected_delivery']}"
    )


order_status_tool = FunctionTool.from_defaults(
    fn=check_order_status,

    name="check_order_status",

    description=(
        "Check a customer's order status using an order ID "
        "such as ORD1001, ORD1002 or ORD1003."
    )
)


# --------------------------------------------------------------
# 13. Tool 2: Calculate product price
# --------------------------------------------------------------

def calculate_final_price(
    unit_price: float,
    quantity: int,
    discount_percentage: float = 0,
    tax_percentage: float = 18
) -> Dict[str, Any]:
    """
    Calculate price after discount and tax.

    Args:
        unit_price: Price of one item.
        quantity: Number of items.
        discount_percentage: Discount percentage.
        tax_percentage: Tax percentage.

    Returns:
        Price calculation details.
    """

    if unit_price < 0:
        raise ValueError(
            "Unit price cannot be negative."
        )

    if quantity <= 0:
        raise ValueError(
            "Quantity must be greater than zero."
        )

    if not 0 <= discount_percentage <= 100:
        raise ValueError(
            "Discount must be between 0 and 100."
        )

    if tax_percentage < 0:
        raise ValueError(
            "Tax percentage cannot be negative."
        )

    subtotal = unit_price * quantity

    discount_amount = (
        subtotal * discount_percentage / 100
    )

    amount_after_discount = (
        subtotal - discount_amount
    )

    tax_amount = (
        amount_after_discount * tax_percentage / 100
    )

    final_price = (
        amount_after_discount + tax_amount
    )

    return {
        "subtotal": round(subtotal, 2),

        "discount_percentage":
            round(discount_percentage, 2),

        "discount_amount":
            round(discount_amount, 2),

        "amount_after_discount":
            round(amount_after_discount, 2),

        "tax_percentage":
            round(tax_percentage, 2),

        "tax_amount":
            round(tax_amount, 2),

        "final_price":
            round(final_price, 2)
    }


price_calculator_tool = FunctionTool.from_defaults(
    fn=calculate_final_price,

    name="calculate_final_price",

    description=(
        "Calculate the final cost of products using unit price, "
        "quantity, discount percentage and tax percentage."
    )
)


In [23]:
# --------------------------------------------------------------
# 14. Tool 3: Create a support ticket
# --------------------------------------------------------------

support_tickets = []


def create_support_ticket(
    customer_name: str,
    issue: str,
    priority: str = "medium"
) -> str:
    """
    Create a customer-support ticket.

    Args:
        customer_name: Customer's name.
        issue: Description of the issue.
        priority: low, medium or high.

    Returns:
        Ticket confirmation.
    """

    normalized_priority = priority.strip().lower()

    valid_priorities = {
        "low",
        "medium",
        "high"
    }

    if normalized_priority not in valid_priorities:
        return (
            "Priority must be low, medium or high."
        )

    if not customer_name.strip():
        return "Customer name cannot be empty."

    if not issue.strip():
        return "Issue description cannot be empty."

    ticket_number = len(support_tickets) + 1

    ticket_id = f"TKT{ticket_number:04d}"

    ticket = {
        "ticket_id": ticket_id,

        "customer_name":
            customer_name.strip(),

        "issue":
            issue.strip(),

        "priority":
            normalized_priority,

        "status":
            "Open",

        "created_at":
            datetime.now().strftime(
                "%d %B %Y, %I:%M %p"
            )
    }

    support_tickets.append(ticket)

    return (
        "Support ticket created successfully.\n"
        f"Ticket ID: {ticket_id}\n"
        f"Customer: {ticket['customer_name']}\n"
        f"Priority: {normalized_priority.title()}\n"
        f"Status: Open\n"
        f"Created at: {ticket['created_at']}"
    )


ticket_creation_tool = FunctionTool.from_defaults(
    fn=create_support_ticket,

    name="create_support_ticket",

    description=(
        "Create a customer-support ticket. The tool requires "
        "the customer's name, issue description and priority."
    )
)


# --------------------------------------------------------------
# 15. Tool 4: List created support tickets
# --------------------------------------------------------------

def list_support_tickets() -> str:
    """
    List all tickets created during this notebook session.
    """

    if not support_tickets:
        return (
            "No support tickets have been created."
        )

    formatted_tickets = []

    for ticket in support_tickets:

        formatted_tickets.append(
            f"{ticket['ticket_id']} | "
            f"{ticket['customer_name']} | "
            f"{ticket['priority'].title()} | "
            f"{ticket['status']} | "
            f"{ticket['issue']}"
        )

    return "\n".join(formatted_tickets)


list_tickets_tool = FunctionTool.from_defaults(
    fn=list_support_tickets,

    name="list_support_tickets",

    description=(
        "List all customer-support tickets created during "
        "the current notebook session."
    )
)


# --------------------------------------------------------------
# 16. Create the Gemini tool-using LlamaIndex agent
# --------------------------------------------------------------

support_agent = FunctionAgent(

    name="GeminiCustomerSupportAgent",

    description=(
        "A Gemini-powered customer-support agent that retrieves "
        "company information from ChromaDB and uses tools."
    ),

    system_prompt="""
You are an intelligent customer-support assistant powered by
Google Gemini.

You have access to the following tools:

1. Company knowledge-base search
2. Order-status lookup
3. Price calculator
4. Support-ticket creation
5. Support-ticket listing

Instructions:

- Use the company knowledge-base tool for troubleshooting,
  warranty, password, billing, refund and working-hours questions.

- Do not invent company policies.

- Use the order-status tool when an order ID is provided.

- Ask the user for an order ID if it is missing.

- Use the price-calculation tool for all price, discount and
  tax calculations.

- Before creating a support ticket, ensure that the customer
  name and issue description are available.

- Explain tool results clearly.

- If information is unavailable, say so honestly.
""",

    tools=[
        knowledge_base_tool,
        order_status_tool,
        price_calculator_tool,
        ticket_creation_tool,
        list_tickets_tool
    ],

    llm=gemini_llm
)




In [24]:
# --------------------------------------------------------------
# 17. Display application instructions
# --------------------------------------------------------------

print("=" * 78)
print("GEMINI + LLAMAINDEX + CHROMADB CUSTOMER-SUPPORT AGENT")
print("=" * 78)

print(
    """
The agent can:

1. Answer questions from a ChromaDB knowledge base.
2. Troubleshoot router and internet problems.
3. Explain warranty, refund and billing policies.
4. Check simulated customer orders.
5. Calculate prices, discounts and taxes.
6. Create support tickets.
7. List created tickets.

Example questions:

- What should I do if the router red light is blinking?

- What does the product warranty cover?

- Check order ORD1001.

- Calculate the total price of three routers costing ₹5,000
  each, with a 10 percent discount and 18 percent tax.

- Create a high-priority support ticket for Ravi because his
  router remains offline after restarting it.

- List all support tickets.

Type exit to stop the program.
"""
)

GEMINI + LLAMAINDEX + CHROMADB CUSTOMER-SUPPORT AGENT

The agent can:

1. Answer questions from a ChromaDB knowledge base.
2. Troubleshoot router and internet problems.
3. Explain warranty, refund and billing policies.
4. Check simulated customer orders.
5. Calculate prices, discounts and taxes.
6. Create support tickets.
7. List created tickets.

Example questions:

- What should I do if the router red light is blinking?

- What does the product warranty cover?

- Check order ORD1001.

- Calculate the total price of three routers costing ₹5,000
  each, with a 10 percent discount and 18 percent tax.

- Create a high-priority support ticket for Ravi because his
  router remains offline after restarting it.

- List all support tickets.

Type exit to stop the program.



In [25]:
while True:
    question = input("You: ").strip()

    if question.lower() in {"exit", "quit", "stop"}:
        print("\nApplication stopped.")
        break

    if not question:
        print("Please enter a valid question.\n")
        continue

    try:
        # Synchronous LlamaIndex query: do not use await
        response = query_engine.query(question)

        print("\nAnswer:")
        print(str(response))

        print("\nRetrieved sources:")

        for source_number, source_node in enumerate(
            response.source_nodes,
            start=1
        ):
            score = source_node.score

            score_text = (
                f"{score:.4f}"
                if score is not None
                else "Not available"
            )

            source_text = (
                source_node.node
                .get_content()
                .strip()
                .replace("\n", " ")
            )

            print(
                f"\nSource {source_number} "
                f"— similarity score: {score_text}"
            )
            print(source_text[:400])

        print("\n" + "=" * 70 + "\n")

    except Exception as error:
        print("\nThe request could not be completed.")
        print(f"{type(error).__name__}: {error}\n")

You: What is warranty?

Answer:
The provided information details a product warranty policy for routers and modems, including its duration, requirements for replacement, and exclusions. However, it does not define what a warranty is.

Retrieved sources:

Source 1 — similarity score: 0.7102
Product Warranty Policy          Routers and modems include a one-year warranty from the         original date of purchase.          Customers must provide the original invoice and the         device serial number for warranty replacement.          Physical damage, water damage and unauthorized repairs         are not covered by the warranty.

Source 2 — similarity score: 0.7102
Product Warranty Policy          Routers and modems include a one-year warranty from the         original date of purchase.          Customers must provide the original invoice and the         device serial number for warranty replacement.          Physical damage, water damage and unauthorized repairs         are not covered 